In [ ]:
! pip install pandas numpy matplotlib seaborn scikit-learn scipy xgboost

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import SpectralClustering
from sklearn.ensemble import IsolationForest, GradientBoostingClassifier
import xgboost as xgb
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import os
import re
import warnings
warnings.filterwarnings('ignore')

In [39]:
def parse_signal_values(signal_str):
    if not signal_str or pd.isna(signal_str):
        return []
    
    signal_str = signal_str.strip('"\'')
    
    try:
        values = [float(val) for val in signal_str.split(',') if val.strip()]
        return values
    except:
        pattern = r'[-+]?\d*\.\d+|\d+'
        values = re.findall(pattern, signal_str)
        values = [float(val) for val in values]
        return values

In [40]:
def load_data(file_path, sample_size=10000):
    try:
        df = pd.read_csv(file_path)
        if sample_size and len(df) > sample_size:
            df = df.sample(sample_size, random_state=42)
        return df
    except Exception as e:
        try:
            alt_path = file_path.replace('\\', '/')
            df = pd.read_csv(alt_path)
            if sample_size and len(df) > sample_size:
                df = df.sample(sample_size, random_state=42)
            return df
        except:
            try:
                filename = os.path.basename(file_path)
                df = pd.read_csv(filename)
                if sample_size and len(df) > sample_size:
                    df = df.sample(sample_size, random_state=42)
                return df
            except:
                return pd.DataFrame()

In [41]:
def preprocess_data(df):
    processed_data = []
    
    for idx, row in df.iterrows():
        try:
            a_current = parse_signal_values(row.get('A Current', ''))
            a_voltage = parse_signal_values(row.get('A Voltage', ''))
            b_current = parse_signal_values(row.get('B Current', ''))
            b_voltage = parse_signal_values(row.get('B Voltage', ''))
            
            row_dict = {
                'Time': row.get('Time', ''),
                'Site_Name': row.get('Site Name', ''),
                'Point_Machine_Name': row.get('Point Machine Name', ''),
                'Direction': row.get('Direction', ''),
                'Type_of_A': row.get('Type of A', ''),
                'Type_of_B': row.get('Type of B', ''),
                'Polling_of_A': row.get('Polling of A', ''),
                'Polling_of_B': row.get('Polling of B', ''),
                'A_Current_Values': a_current,
                'A_Voltage_Values': a_voltage,
                'B_Current_Values': b_current,
                'B_Voltage_Values': b_voltage
            }
            
            processed_data.append(row_dict)
        except:
            continue
    
    return pd.DataFrame(processed_data)

In [42]:
def calculate_statistics(values):
    if not values or len(values) == 0:
        return {
            'mean': 0, 'std': 0, 'min': 0, 'max': 0, 'range': 0,
            'slope': 0, 'peak_count': 0, 'peak_avg': 0, 'zero_crossings': 0
        }
    
    values = np.array(values)
    mean = np.mean(values)
    std = np.std(values)
    min_val = np.min(values)
    max_val = np.max(values)
    range_val = max_val - min_val
    
    # Calculate slope
    x = np.arange(len(values))
    if len(values) > 1:
        slope = np.polyfit(x, values, 1)[0]
    else:
        slope = 0
    
    # Count peaks
    peaks = []
    for i in range(1, len(values) - 1):
        if values[i] > values[i-1] and values[i] > values[i+1]:
            peaks.append(values[i])
    
    peak_count = len(peaks)
    peak_avg = np.mean(peaks) if peak_count > 0 else 0
    
    # Count zero crossings
    zero_crossings = 0
    for i in range(1, len(values)):
        if (values[i-1] > 0 and values[i] < 0) or (values[i-1] < 0 and values[i] > 0):
            zero_crossings += 1
    
    return {
        'mean': mean,
        'std': std,
        'min': min_val,
        'max': max_val,
        'range': range_val,
        'slope': slope,
        'peak_count': peak_count,
        'peak_avg': peak_avg,
        'zero_crossings': zero_crossings
    }

In [43]:
def calculate_correlation(signal1, signal2):
    if len(signal1) < 2 or len(signal2) < 2:
        return 0
    
    min_length = min(len(signal1), len(signal2))
    signal1 = signal1[:min_length]
    signal2 = signal2[:min_length]
    
    if np.std(signal1) == 0 or np.std(signal2) == 0:
        return 0
    
    return np.corrcoef(signal1, signal2)[0, 1]

In [44]:
def extract_features(df):
    features_list = []
    
    for idx, row in df.iterrows():
        try:
            feature_dict = {
                'Time': row.get('Time', ''),
                'Site_Name': row.get('Site_Name', ''),
                'Point_Machine_Name': row.get('Point_Machine_Name', ''),
                'Direction': row.get('Direction', ''),
                'Type_of_A': row.get('Type_of_A', ''),
                'Type_of_B': row.get('Type_of_B', ''),
                'Polling_of_A': row.get('Polling_of_A', ''),
                'Polling_of_B': row.get('Polling_of_B', '')
            }
            
            # A Current features
            a_current_stats = calculate_statistics(row['A_Current_Values'])
            for key, value in a_current_stats.items():
                feature_dict[f'A_Current_{key}'] = value
            
            # A Voltage features
            a_voltage_stats = calculate_statistics(row['A_Voltage_Values'])
            for key, value in a_voltage_stats.items():
                feature_dict[f'A_Voltage_{key}'] = value
            
            # B Current features
            b_current_stats = calculate_statistics(row['B_Current_Values'])
            for key, value in b_current_stats.items():
                feature_dict[f'B_Current_{key}'] = value
            
            # B Voltage features
            b_voltage_stats = calculate_statistics(row['B_Voltage_Values'])
            for key, value in b_voltage_stats.items():
                feature_dict[f'B_Voltage_{key}'] = value
            
            # Calculate derived features
            if a_current_stats['mean'] > 0:
                feature_dict['A_Impedance'] = a_voltage_stats['mean'] / a_current_stats['mean']
            else:
                feature_dict['A_Impedance'] = 0
                
            if b_current_stats['mean'] > 0:
                feature_dict['B_Impedance'] = b_voltage_stats['mean'] / b_current_stats['mean']
            else:
                feature_dict['B_Impedance'] = 0
            
            # Power-like product
            feature_dict['A_Power'] = a_voltage_stats['mean'] * a_current_stats['mean']
            feature_dict['B_Power'] = b_voltage_stats['mean'] * b_current_stats['mean']
            
            # Correlation
            feature_dict['A_Current_Voltage_Corr'] = calculate_correlation(row['A_Current_Values'], row['A_Voltage_Values'])
            feature_dict['B_Current_Voltage_Corr'] = calculate_correlation(row['B_Current_Values'], row['B_Voltage_Values'])
            
            features_list.append(feature_dict)
                
        except:
            continue
    
    return pd.DataFrame(features_list)

In [45]:
def perform_eda(features_df):
    numerical_cols = features_df.select_dtypes(include=['number']).columns.tolist()
    
    plt.figure(figsize=(15, 10))
    
    # Distribution of key numerical features
    key_features = ['A_Current_mean', 'A_Voltage_mean', 'B_Current_mean', 'B_Voltage_mean', 
                    'A_Impedance', 'B_Impedance', 'A_Power', 'B_Power']
    
    key_features = [f for f in key_features if f in features_df.columns]
    
    for i, col in enumerate(key_features[:4]):
        plt.subplot(2, 2, i+1)
        sns.histplot(features_df[col].dropna(), kde=True)
        plt.title(f'Distribution of {col}')
    
    plt.tight_layout()
    plt.show()
    
    return features_df

In [46]:
def apply_spectral_clustering(features_df, n_clusters=5):
    numeric_cols = features_df.select_dtypes(include=['number']).columns
    X = features_df[numeric_cols].fillna(0).values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    spectral = SpectralClustering(
        n_clusters=n_clusters,
        eigen_solver='arpack',
        affinity="nearest_neighbors",
        n_neighbors=min(10, len(X_scaled)-1),
        random_state=42
    )
    
    cluster_labels = spectral.fit_predict(X_scaled)
    
    result_df = features_df.copy()
    result_df['cluster'] = cluster_labels
    
    if len(np.unique(cluster_labels)) > 1:
        silhouette_avg = silhouette_score(X_scaled, cluster_labels)
        print(f"Silhouette Score: {silhouette_avg:.4f}")
    
    plt.figure(figsize=(10, 5))
    cluster_counts = result_df['cluster'].value_counts().sort_index()
    sns.barplot(x=cluster_counts.index, y=cluster_counts.values)
    plt.title('Cluster Distribution')
    plt.xlabel('Cluster')
    plt.ylabel('Count')
    plt.show()
    
    return result_df, X_scaled, cluster_labels

In [47]:
def detect_anomalies_isolation_forest(X_scaled, contamination=0.05):
    iso_forest = IsolationForest(
        n_estimators=100,
        contamination=contamination,
        random_state=42
    )
    
    anomaly_labels = iso_forest.fit_predict(X_scaled)
    
    return anomaly_labels

In [48]:
def detect_anomalies_spectral(X_scaled, contamination=0.05):
    from sklearn.neighbors import NearestNeighbors
    
    k = min(int(X_scaled.shape[0] * 0.05) + 1, X_scaled.shape[0]-1)
    nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled)
    distances, indices = nbrs.kneighbors(X_scaled)
    
    avg_distances = distances[:, 1:].mean(axis=1)
    
    normalized_distances = (avg_distances - np.min(avg_distances)) / (np.max(avg_distances) - np.min(avg_distances) + 1e-10)
    
    threshold = np.percentile(normalized_distances, 100 * (1 - contamination))
    anomaly_labels = np.ones(X_scaled.shape[0], dtype=int)
    anomaly_labels[normalized_distances > threshold] = -1
    
    return anomaly_labels

In [49]:
def train_gradient_boost(features_df, cluster_labels):
    numeric_cols = features_df.select_dtypes(include=['number']).columns
    X = features_df[numeric_cols].fillna(0).values
    y = cluster_labels
    
    gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
    gb_model.fit(X, y)
    
    importances = pd.Series(gb_model.feature_importances_, index=numeric_cols)
    top_importances = importances.sort_values(ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_importances.values, y=top_importances.index)
    plt.title('Top 10 Important Features from Gradient Boosting')
    plt.tight_layout()
    plt.show()
    
    return gb_model, importances

In [50]:
def train_xgboost(features_df, cluster_labels):
    numeric_cols = features_df.select_dtypes(include=['number']).columns
    X = features_df[numeric_cols].fillna(0).values
    y = cluster_labels
    
    xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=42)
    xgb_model.fit(X, y)
    
    importances = pd.Series(xgb_model.feature_importances_, index=numeric_cols)
    top_importances = importances.sort_values(ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_importances.values, y=top_importances.index)
    plt.title('Top 10 Important Features from XGBoost')
    plt.tight_layout()
    plt.show()
    
    return xgb_model, importances

In [51]:
def calculate_additional_metrics(X_scaled, cluster_labels):
    db_score = davies_bouldin_score(X_scaled, cluster_labels)
    ch_score = calinski_harabasz_score(X_scaled, cluster_labels)
    
    print("\nClustering Evaluation Metrics:")
    print(f"Davies-Bouldin Index: {db_score:.4f} (lower is better)")
    print(f"Calinski-Harabasz Index: {ch_score:.4f} (higher is better)")
    
    return {
        'davies_bouldin': db_score,
        'calinski_harabasz': ch_score
    }

In [52]:
def analyze_anomalies(clustered_df, X_scaled, anomaly_labels):
    clustered_df['anomaly'] = anomaly_labels
    clustered_df['is_anomaly'] = (anomaly_labels == -1).astype(int)
    
    anomaly_count = clustered_df['is_anomaly'].sum()
    anomaly_percentage = (anomaly_count / len(clustered_df)) * 100
    print(f"Detected {anomaly_count} anomalies ({anomaly_percentage:.2f}%)")
    
    plt.figure(figsize=(10, 5))
    anomaly_by_cluster = clustered_df.groupby('cluster')['is_anomaly'].mean() * 100
    sns.barplot(x=anomaly_by_cluster.index, y=anomaly_by_cluster.values)
    plt.title('Percentage of Anomalies by Cluster')
    plt.xlabel('Cluster')
    plt.ylabel('Anomaly Percentage (%)')
    plt.show()
    
    return clustered_df

In [53]:
def analyze_clusters(clustered_df):
    cluster_stats = clustered_df.groupby('cluster').agg({
        'A_Current_mean': ['mean', 'std'],
        'A_Voltage_mean': ['mean', 'std'],
        'B_Current_mean': ['mean', 'std'],
        'B_Voltage_mean': ['mean', 'std'],
        'A_Impedance': ['mean', 'std'],
        'B_Impedance': ['mean', 'std'],
        'A_Power': ['mean', 'std'],
        'B_Power': ['mean', 'std'],
        'is_anomaly': ['sum', 'mean']
    })
    
    print("\nCluster Statistics:")
    print(cluster_stats)
    
    return cluster_stats

In [54]:
def process_dataset(file_path, sample_size=10000, n_clusters=5):
    df = load_data(file_path, sample_size)
    
    if df.empty:
        print("Dataset loading failed.")
        return None
    
    processed_df = preprocess_data(df)
    features_df = extract_features(processed_df)
    features_df = perform_eda(features_df)
    
    clustered_df, X_scaled, cluster_labels = apply_spectral_clustering(features_df, n_clusters)
    
    gb_model, gb_importances = train_gradient_boost(features_df, cluster_labels)
    xgb_model, xgb_importances = train_xgboost(features_df, cluster_labels)
    
    metrics = calculate_additional_metrics(X_scaled, cluster_labels)
    
    iso_anomaly_labels = detect_anomalies_isolation_forest(X_scaled)
    clustered_df = analyze_anomalies(clustered_df, X_scaled, iso_anomaly_labels)
    
    spectral_anomaly_labels = detect_anomalies_spectral(X_scaled)
    clustered_df['spectral_anomaly'] = (spectral_anomaly_labels == -1).astype(int)
    
    from collections import Counter
    agreement = (clustered_df['is_anomaly'] == clustered_df['spectral_anomaly']).mean() * 100
    print(f"Agreement between anomaly detection methods: {agreement:.2f}%")
    
    cluster_stats = analyze_clusters(clustered_df)
    
    return {
        'clustered_df': clustered_df,
        'features_df': features_df,
        'X_scaled': X_scaled,
        'cluster_labels': cluster_labels,
        'metrics': metrics,
        'gb_model': gb_model,
        'gb_importances': gb_importances,
        'xgb_model': xgb_model,
        'xgb_importances': xgb_importances,
        'cluster_stats': cluster_stats
    }

In [55]:
def generate_report(results):
    clustered_df = results['clustered_df']
    features_df = results['features_df']
    cluster_stats = results['cluster_stats']
    
    print("\n" + "="*50)
    print(" ANALYSIS REPORT ")
    print("="*50)
    
    print("\n1. DATASET OVERVIEW")
    print(f"- Total records analyzed: {len(clustered_df)}")
    print(f"- Total features extracted: {features_df.shape[1]}")
    
    print("\n2. CLUSTERING RESULTS")
    print(f"- Number of clusters: {len(clustered_df['cluster'].unique())}")
    
    cluster_counts = clustered_df['cluster'].value_counts().sort_index()
    for cluster, count in cluster_counts.items():
        percentage = count / len(clustered_df) * 100
        print(f"  * Cluster {cluster}: {count} records ({percentage:.2f}%)")
    
    print("\n3. ANOMALY DETECTION RESULTS")
    
    if_anomalies = clustered_df['is_anomaly'].sum()
    if_pct = if_anomalies / len(clustered_df) * 100
    print(f"- Isolation Forest detected {if_anomalies} anomalies ({if_pct:.2f}%)")
    
    if 'spectral_anomaly' in clustered_df.columns:
        sp_anomalies = clustered_df['spectral_anomaly'].sum()
        sp_pct = sp_anomalies / len(clustered_df) * 100
        print(f"- Spectral method detected {sp_anomalies} anomalies ({sp_pct:.2f}%)")
    
    print("\n- Anomaly distribution across clusters:")
    anomaly_by_cluster = clustered_df.groupby('cluster')['is_anomaly'].agg(['sum', 'mean'])
    anomaly_by_cluster['percentage'] = anomaly_by_cluster['mean'] * 100
    print(anomaly_by_cluster[['sum', 'percentage']])
    
    print("\n4. KEY FEATURES FOR CLUSTERING")
    gb_importances = results['gb_importances']
    xgb_importances = results['xgb_importances']
    
    top_gb = gb_importances.sort_values(ascending=False).head(5)
    top_xgb = xgb_importances.sort_values(ascending=False).head(5)
    
    print("Top important features from Gradient Boosting:")
    for feature, importance in top_gb.items():
        print(f"  * {feature}: {importance:.4f}")
    
    print("\nTop important features from XGBoost:")
    for feature, importance in top_xgb.items():
        print(f"  * {feature}: {importance:.4f}")
    
    print("\n" + "="*50)
    print(" END OF REPORT ")
    print("="*50)

In [56]:
def run_analysis(file_path, sample_size=10000, n_clusters=5):
    print(f"Analyzing file: {file_path}")
    print(f"Sample size: {sample_size}")
    print(f"Number of clusters: {n_clusters}")
    
    results = process_dataset(file_path, sample_size, n_clusters)
    
    if results is None:
        print("Analysis failed.")
        return None
    
    generate_report(results)
    
    return results

In [57]:
datasets = {
    'CR': r'FEB 2025 Prototype data\CR.csv',
    'ECOR': r'FEB 2025 Prototype data\ECOR.csv',
    'ECR': r'FEB 2025 Prototype data\ECR.csv',
    'NCR': r'FEB 2025 Prototype data\NCR.csv',
    'NER': r'FEB 2025 Prototype data\NER.csv',
    'NFR': r'FEB 2025 Prototype data\NFR.csv',
    'NR': r'FEB 2025 Prototype data\NR.csv',
    'SCR': r'FEB 2025 Prototype data\SCR.csv',
    'SECR': r'FEB 2025 Prototype data\SECR.csv',
    'SR': r'FEB 2025 Prototype data\SR.csv',
    'WCR': r'FEB 2025 Prototype data\WCR.csv',
    'WR': r'FEB 2025 Prototype data\WR.csv',
    'TEST_ZONE': r'FEB 2025 Prototype data\Test_Zone.csv'
}

In [58]:
alternative_datasets = {
    'CR': 'CR.csv',
    'ECOR': 'ECOR.csv',
    'ECR': 'ECR.csv',
    'NCR': 'NCR.csv',
    'NER': 'NER.csv',
    'NFR': 'NFR.csv',
    'NR': 'NR.csv',
    'SCR': 'SCR.csv',
    'SECR': 'SECR.csv',
    'SR': 'SR.csv',
    'WCR': 'WCR.csv',
    'WR': 'WR.csv',
    'TEST_ZONE': 'Test_Zone.csv'
}

In [ ]:
chosen_dataset = 'TEST_ZONE'
file_path = datasets[chosen_dataset]

try:
    results = run_analysis(file_path, sample_size=10000, n_clusters=5)
except FileNotFoundError:
    try:
        alt_path = file_path.replace('\\', '/')
        results = run_analysis(alt_path, sample_size=10000, n_clusters=5)
    except FileNotFoundError:
        file_path = alternative_datasets[chosen_dataset]
        results = run_analysis(file_path, sample_size=10000, n_clusters=5)

In [60]:
def create_correlation_heatmap(features_df):
    numeric_cols = features_df.select_dtypes(include=['number']).columns
    important_features = [
        'A_Current_mean', 'A_Current_std', 'A_Current_max', 
        'A_Voltage_mean', 'A_Voltage_std', 'A_Voltage_max',
        'B_Current_mean', 'B_Current_std', 'B_Current_max',
        'B_Voltage_mean', 'B_Voltage_std', 'B_Voltage_max',
        'A_Impedance', 'B_Impedance', 'A_Power', 'B_Power',
        'A_Current_Voltage_Corr', 'B_Current_Voltage_Corr'
    ]
    existing_features = [col for col in important_features if col in numeric_cols]
    correlation_matrix = features_df[existing_features].corr()
    
    plt.figure(figsize=(14, 10))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
    plt.title('Correlation Heatmap of Key Features')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    return correlation_matrix

In [61]:
def create_cluster_feature_heatmap(clustered_df):
    features = [
        'A_Current_mean', 'A_Voltage_mean', 'A_Impedance', 'A_Power',
        'B_Current_mean', 'B_Voltage_mean', 'B_Impedance', 'B_Power'
    ]
    features = [f for f in features if f in clustered_df.columns]
    pivot_df = clustered_df.pivot_table(index='cluster', values=features)
    scaler = StandardScaler()
    pivot_scaled = pd.DataFrame(
        scaler.fit_transform(pivot_df),
        index=pivot_df.index,
        columns=pivot_df.columns
    )
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_scaled, annot=True, cmap='viridis', fmt=".2f", linewidths=0.5)
    plt.title('Standardized Feature Means by Cluster')
    plt.tight_layout()
    plt.show()

In [62]:
def create_cluster_feature_heatmap(clustered_df):
    features = [
        'A_Current_mean', 'A_Voltage_mean', 'A_Impedance', 'A_Power',
        'B_Current_mean', 'B_Voltage_mean', 'B_Impedance', 'B_Power'
    ]
    features = [f for f in features if f in clustered_df.columns]
    pivot_df = clustered_df.pivot_table(index='cluster', values=features)
    scaler = StandardScaler()
    pivot_scaled = pd.DataFrame(
        scaler.fit_transform(pivot_df),
        index=pivot_df.index,
        columns=pivot_df.columns
    )
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_scaled, annot=True, cmap='viridis', fmt=".2f", linewidths=0.5)
    plt.title('Standardized Feature Means by Cluster')
    plt.tight_layout()
    plt.show()

In [63]:
def create_pca_visualization(X_scaled, labels, title='PCA Visualization'):
    from sklearn.decomposition import PCA
    
    # Apply PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # Plot the results
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.7, s=30)
    plt.colorbar(scatter)
    plt.title(title)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    return X_pca

In [64]:
def evaluate_clustering_quality(X_scaled, cluster_labels):
    from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score, silhouette_score
    
    silhouette = silhouette_score(X_scaled, cluster_labels)
    db = davies_bouldin_score(X_scaled, cluster_labels)
    ch = calinski_harabasz_score(X_scaled, cluster_labels)
    
    print("\nCLUSTERING QUALITY METRICS:")
    print(f"Silhouette Score: {silhouette:.4f} (higher is better, range: [-1, 1])")
    print(f"Davies-Bouldin Index: {db:.4f} (lower is better)")
    print(f"Calinski-Harabasz Index: {ch:.4f} (higher is better)")
    
    return {
        'silhouette': silhouette,
        'davies_bouldin': db,
        'calinski_harabasz': ch
    }

In [65]:
def plot_feature_importances_comparison(gb_importances, xgb_importances):
    # Combine importances
    importances_df = pd.DataFrame({
        'Gradient Boosting': gb_importances,
        'XGBoost': xgb_importances
    })
    
    # Get top 10 features by combined importance
    importances_df['Combined'] = importances_df.sum(axis=1)
    top_features = importances_df.sort_values('Combined', ascending=False).head(10).index
    
    # Plot comparison
    plt.figure(figsize=(12, 8))
    importances_df.loc[top_features, ['Gradient Boosting', 'XGBoost']].plot(kind='bar')
    plt.title('Feature Importance Comparison: Gradient Boosting vs XGBoost')
    plt.xlabel('Features')
    plt.ylabel('Importance')
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [66]:
def create_boxplots_by_cluster(clustered_df):
    features = [
        'A_Current_mean', 'A_Voltage_mean', 
        'A_Impedance', 'A_Power'
    ]
    
    # Filter existing features
    features = [f for f in features if f in clustered_df.columns]
    
    # Create boxplots
    plt.figure(figsize=(15, 10))
    
    for i, feature in enumerate(features):
        plt.subplot(2, 2, i+1)
        sns.boxplot(x='cluster', y=feature, data=clustered_df)
        plt.title(f'{feature} by Cluster')
        plt.xlabel('Cluster')
        plt.ylabel(feature)
    
    plt.tight_layout()
    plt.show()

In [67]:
def create_radar_chart(clustered_df):
    import matplotlib.pyplot as plt
    from matplotlib.path import Path
    from matplotlib.projections import register_projection
    from matplotlib.projections.polar import PolarAxes
    from matplotlib.spines import Spine
    
    def radar_factory(num_vars, frame='circle'):
        theta = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
        
        class RadarAxes(PolarAxes):
            name = 'radar'
            
            def __init__(self, *args, **kwargs):
                super().__init__(*args, **kwargs)
                self.set_theta_zero_location('N')
                
            def fill(self, *args, **kwargs):
                return super().fill_between(*args, **kwargs)
                
            def plot(self, *args, **kwargs):
                lines = super().plot(*args, **kwargs)
                self._close_polygon(lines[0])
                return lines
                
            def _close_polygon(self, line):
                x, y = line.get_data()
                # Draw the polygon by closing it
                if x[0] != x[-1]:
                    x = np.concatenate((x, [x[0]]))
                    y = np.concatenate((y, [y[0]]))
                    line.set_data(x, y)
                    
            def set_varlabels(self, labels):
                self.set_thetagrids(np.degrees(theta), labels)
                
        register_projection(RadarAxes)
        return theta

    # Get the most important features
    features = [
        'A_Current_mean', 'A_Voltage_mean', 'A_Current_std', 'A_Voltage_std',
        'A_Impedance', 'A_Power'
    ]
    
    # Keep only features that exist in the dataframe
    features = [f for f in features if f in clustered_df.columns]
    
    if len(features) < 3:
        print("Not enough features for radar chart")
        return
    
    # Calculate means for each cluster
    cluster_means = clustered_df.groupby('cluster')[features].mean()
    
    # Normalize values to [0,1] for radar chart
    scaler = StandardScaler()
    scaled_means = scaler.fit_transform(cluster_means)
    scaled_means = (scaled_means - scaled_means.min(axis=0)) / (scaled_means.max(axis=0) - scaled_means.min(axis=0) + 1e-10)
    
    # Create the radar chart
    theta = radar_factory(len(features))
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='radar'))
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(cluster_means)))
    
    for i, (idx, row) in enumerate(zip(cluster_means.index, scaled_means)):
        ax.plot(theta, row, color=colors[i], label=f'Cluster {idx}')
        ax.fill(theta, row, color=colors[i], alpha=0.1)
    
    ax.set_varlabels(features)
    plt.legend(loc='upper right')
    plt.title('Cluster Profiles Radar Chart')
    plt.show()

In [68]:
def analyze_confusion_matrix(clustered_df):
    # Create a confusion matrix between the two anomaly detection methods
    if 'is_anomaly' not in clustered_df.columns or 'spectral_anomaly' not in clustered_df.columns:
        print("Missing anomaly detection results")
        return
    
    confusion_matrix = pd.crosstab(
        clustered_df['is_anomaly'], 
        clustered_df['spectral_anomaly'],
        rownames=['Isolation Forest'],
        colnames=['Spectral Method']
    )
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues')
    plt.title('Comparison of Anomaly Detection Methods')
    plt.tight_layout()
    plt.show()
    
    # Calculate metrics
    total = len(clustered_df)
    agreement = (clustered_df['is_anomaly'] == clustered_df['spectral_anomaly']).sum() / total * 100
    both_anomalies = ((clustered_df['is_anomaly'] == 1) & (clustered_df['spectral_anomaly'] == 1)).sum()
    either_anomalies = ((clustered_df['is_anomaly'] == 1) | (clustered_df['spectral_anomaly'] == 1)).sum()
    
    print("\nANOMALY DETECTION COMPARISON:")
    print(f"Agreement between methods: {agreement:.2f}%")
    print(f"Records flagged as anomalies by both methods: {both_anomalies}")
    print(f"Records flagged as anomalies by either method: {either_anomalies}")
    
    if either_anomalies > 0:
        print(f"Jaccard similarity (intersection over union): {both_anomalies/either_anomalies:.4f}")

In [69]:
def visualize_time_series_samples(processed_df, clustered_df, n_samples=2):
    # Select samples from each cluster
    clusters = clustered_df['cluster'].unique()
    
    plt.figure(figsize=(15, n_samples * len(clusters) * 3))
    plot_idx = 1
    
    for cluster in clusters:
        cluster_indices = clustered_df[clustered_df['cluster'] == cluster].index
        
        if len(cluster_indices) < n_samples:
            samples = cluster_indices
        else:
            samples = np.random.choice(cluster_indices, n_samples, replace=False)
        
        for sample_idx in samples:
            row = processed_df.iloc[sample_idx]
            
            # Plot A Current
            plt.subplot(len(clusters) * n_samples, 2, plot_idx)
            plt.plot(row['A_Current_Values'])
            plt.title(f"Cluster {cluster} - A Current")
            plt.xlabel('Time Index')
            plt.ylabel('Current Value')
            
            # Plot A Voltage
            plt.subplot(len(clusters) * n_samples, 2, plot_idx + 1)
            plt.plot(row['A_Voltage_Values'])
            plt.title(f"Cluster {cluster} - A Voltage")
            plt.xlabel('Time Index')
            plt.ylabel('Voltage Value')
            
            plot_idx += 2
    
    plt.tight_layout()
    plt.show()

In [70]:
def enhanced_process_dataset(file_path, sample_size=10000, n_clusters=5):
    df = load_data(file_path, sample_size)
    
    if df.empty:
        print("Dataset loading failed.")
        return None
    
    processed_df = preprocess_data(df)
    features_df = extract_features(processed_df)
    
    # Perform EDA with correlation heatmap
    features_df = perform_eda(features_df)
    correlation_matrix = create_correlation_heatmap(features_df)
    
    # Apply spectral clustering
    clustered_df, X_scaled, cluster_labels = apply_spectral_clustering(features_df, n_clusters)
    
    # Generate additional visualizations
    create_pca_visualization(X_scaled, cluster_labels, 'PCA Visualization of Clusters')
    create_boxplots_by_cluster(clustered_df)
    create_cluster_feature_heatmap(clustered_df)
    
    # Train models
    gb_model, gb_importances = train_gradient_boost(features_df, cluster_labels)
    xgb_model, xgb_importances = train_xgboost(features_df, cluster_labels)
    plot_feature_importances_comparison(gb_importances, xgb_importances)
    
    # Evaluate clustering quality
    metrics = evaluate_clustering_quality(X_scaled, cluster_labels)
    
    # Detect anomalies
    iso_anomaly_labels = detect_anomalies_isolation_forest(X_scaled)
    clustered_df = analyze_anomalies(clustered_df, X_scaled, iso_anomaly_labels)
    
    spectral_anomaly_labels = detect_anomalies_spectral(X_scaled)
    clustered_df['spectral_anomaly'] = (spectral_anomaly_labels == -1).astype(int)
    
    # Compare anomaly detection methods
    analyze_confusion_matrix(clustered_df)
    
    # Show radar chart of cluster profiles
    create_radar_chart(clustered_df)
    
    # Visualize sample time series from each cluster
    visualize_time_series_samples(processed_df, clustered_df)
    
    # Analyze clusters
    cluster_stats = analyze_clusters(clustered_df)
    
    return {
        'processed_df': processed_df,
        'clustered_df': clustered_df,
        'features_df': features_df,
        'X_scaled': X_scaled,
        'cluster_labels': cluster_labels,
        'metrics': metrics,
        'gb_model': gb_model,
        'gb_importances': gb_importances,
        'xgb_model': xgb_model,
        'xgb_importances': xgb_importances,
        'cluster_stats': cluster_stats,
        'correlation_matrix': correlation_matrix
    }

In [71]:
def run_enhanced_analysis(file_path, sample_size=10000, n_clusters=5):
    print(f"Analyzing file: {file_path}")
    print(f"Sample size: {sample_size}")
    print(f"Number of clusters: {n_clusters}")
    
    results = enhanced_process_dataset(file_path, sample_size, n_clusters)
    
    if results is None:
        print("Analysis failed.")
        return None
    
    generate_report(results)
    
    return results

In [ ]:
chosen_dataset = 'CR'
file_path = datasets[chosen_dataset]

try:
    results = run_enhanced_analysis(file_path, sample_size=10000, n_clusters=5)
except FileNotFoundError:
    try:
        alt_path = file_path.replace('\\', '/')
        results = run_enhanced_analysis(alt_path, sample_size=10000, n_clusters=5)
    except FileNotFoundError:
        file_path = alternative_datasets[chosen_dataset]
        results = run_enhanced_analysis(file_path, sample_size=10000, n_clusters=5)